# 03 - ML Training and Preprocessing Artifacts

Notebook này huấn luyện mô hình baseline supervised và đóng gói artifacts để RL warmstart dùng lại.

Đầu ra mục tiêu:
- best_traffic_model_baseline.pt
- preprocessing_artifacts.pkl
- ml_metrics.json


In [1]:
from pathlib import Path
import os
import json
import subprocess

ROOT = Path('/workspace/ai-core')
RUN_SCRIPT = ROOT / 'scripts' / 'run_ml_train.py'
ARTIFACT_ROOT = Path(os.getenv('ML_ARTIFACT_ROOT', ROOT / 'artifacts' / 'ml'))

# ĐƯỜNG DẪN DỮ LIỆU ĐÃ ĐƯỢC BALANCED TỪ BƯỚC 02
INPUT_DATA_PATH = ROOT / 'data' / 'processed' / '02_balanced_training_data.parquet'
os.environ['ML_INPUT_DATA_PATH'] = str(INPUT_DATA_PATH)

print('Run script:', RUN_SCRIPT)
print('Artifact root:', ARTIFACT_ROOT)
print('Input data path:', INPUT_DATA_PATH if INPUT_DATA_PATH.exists() else '❌ NOT FOUND')

Run script: /workspace/ai-core/scripts/run_ml_train.py
Artifact root: /workspace/ai-core/artifacts/ml
Input data path: /workspace/ai-core/data/processed/02_balanced_training_data.parquet


## Cấu hình chạy

Nếu muốn chạy thật trong notebook, bật RUN_TRAIN=True ở cell kế tiếp.
Mặc định cell sẽ ở chế độ dry-run để tránh chạy tốn tài nguyên ngoài ý muốn.


In [2]:
RUN_TRAIN = True
CMD = ['python', str(RUN_SCRIPT)]
print("Command:", " ".join(CMD))
if RUN_TRAIN:
    # Chạy và hiển thị output trực tiếp (streaming)
    import subprocess
    import sys
    
    process = subprocess.Popen(
        CMD, 
        cwd=str(ROOT), 
        stdout=subprocess.PIPE, 
        stderr=subprocess.STDOUT, 
        text=True,
        bufsize=1
    )
    
    print("--- BẮT ĐẦU TRAINING OUTPUT ---")
    for line in process.stdout:
        print(line, end="")
        sys.stdout.flush()
    
    process.wait()
    print(f"--- KẾT THÚC. Exit code: {process.returncode} ---")
else:
    print("Dry-run: chưa thực thi train.")

Command: python /workspace/ai-core/scripts/run_ml_train.py
--- BẮT ĐẦU TRAINING OUTPUT ---
--- KHỞI ĐỘNG HUẤN LUYỆN TOÀN TẬP TRÊN 6 CORRIDORS ---
🧪 Run=manual_h15 | weighted_sampler=False | class_weights=True | clip=[1.0, 1.6] | epochs=50 | lr=0.001 | batch_size=256 | patience=12 | horizon=15m | loss=focal | dropout=0.2 | weight_decay=0.0001 | label_smoothing=0.05 | lr_scheduler=True | window_balancing=True | ckpt=/app/artifacts/ml/checkpoints/best_traffic_model_manual_h15.pt
📦 LOADING BALANCED DATA FROM PARQUET: /workspace/ai-core/data/processed/02_balanced_training_data.parquet

✅ ĐÃ TẢI THÀNH CÔNG SIÊU TẬP DỮ LIỆU: 9935822 dòng.
⏳ Đang tính toán DataLoaders (Quá trình mã hóa và scale có thể mất vài phút)...
Tổng số dòng dữ liệu thô: 9935822
Tổng số cửa sổ hợp lệ thu được: 764294 (window=12, target_offset=1)
Phân bổ Class trong các cửa sổ:
  - Class 0: 205464 windows
  - Class 1: 250000 windows
  - Class 2: 205464 windows
  - Class 3: 51366 windows
  - Class 4: 30000 windows
  - Clas

In [3]:
# Kiểm tra artifact sau khi train
checkpoints = sorted((ARTIFACT_ROOT / "checkpoints").glob("*.pt")) if (ARTIFACT_ROOT / "checkpoints").exists() else []
preps = sorted((ARTIFACT_ROOT / "preprocessing").glob("*.pkl")) if (ARTIFACT_ROOT / "preprocessing").exists() else []
metrics = sorted((ARTIFACT_ROOT / "metrics").glob("*.json")) if (ARTIFACT_ROOT / "metrics").exists() else []
print("Checkpoints:", [p.name for p in checkpoints])
print("Preprocessing:", [p.name for p in preps])
print("Metrics:", [p.name for p in metrics])


Checkpoints: ['best_traffic_model.pt', 'best_traffic_model_baseline.pt', 'best_traffic_model_manual_h15.pt', 'best_traffic_model_manual_h30.pt']
Preprocessing: ['preprocessing_artifacts.pkl', 'preprocessing_artifacts_manual_h15.pkl', 'preprocessing_artifacts_manual_h30.pkl']
Metrics: ['ml_metrics_manual_h15.json', 'ml_metrics_manual_h30.json']


In [4]:
# Tạo symlink/tên chuẩn để warmstart dễ dùng (tuỳ chọn)
baseline_ckpt = checkpoints[0] if checkpoints else None
baseline_prep = preps[0] if preps else None
if baseline_ckpt:
    target = ARTIFACT_ROOT / "checkpoints" / "best_traffic_model_baseline.pt"
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(baseline_ckpt.name)
    print("Linked:", target, "->", baseline_ckpt.name)
if baseline_prep:
    target = ARTIFACT_ROOT / "preprocessing" / "preprocessing_artifacts.pkl"
    if target.exists() or target.is_symlink():
        target.unlink()
    target.symlink_to(baseline_prep.name)
    print("Linked:", target, "->", baseline_prep.name)


Linked: /workspace/ai-core/artifacts/ml/checkpoints/best_traffic_model_baseline.pt -> best_traffic_model.pt
Linked: /workspace/ai-core/artifacts/ml/preprocessing/preprocessing_artifacts.pkl -> preprocessing_artifacts.pkl
